In [ ]:
import os
import torch
import requests
from PIL import Image
from io import BytesIO
from torchvision import transforms
from google.colab import drive

drive.mount('/content/drive')
base_dir = "/content/drive/MyDrive/Colab Notebooks/IOT"
images_dir = os.path.join(base_dir, "dataset_test/images")
images_number = 60

os.makedirs(images_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo di calcolo rilevato: {device}")

# preprocessing DINOv2
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

valid_extensions = ('.png', '.jpg', '.jpeg')
existing_images = [f for f in os.listdir(images_dir) if f.lower().endswith(valid_extensions)]

if len(existing_images) < images_number:
    print("Immagini insufficienti. Download...")
    num_images_to_download = images_number - len(existing_images)
    for i in range(num_images_to_download):
        url = f"https://picsum.photos/224/224?random={i}"
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content)).convert("RGB")
                img_path = os.path.join(images_dir, f"test_img_{i + len(existing_images):02d}.jpg")
                img.save(img_path)
        except Exception as e:
            print(f"Errore download: {e}")

    existing_images = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(valid_extensions)])
else:
    print(f"Rilevate {len(existing_images)} immagini.")
    existing_images = sorted(existing_images)

image_tensors = []
for f in existing_images[:images_number]:
    img_path = os.path.join(images_dir, f)
    try:
        tensor = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
        image_tensors.append(tensor)
    except Exception as e:
        print(f"Errore nel pre-processing del file {f}: {e}")

print(f"[OK] {len(image_tensors)} tensori pronti su {device}!")

In [ ]:
import pandas as pd
import time
import psutil
import os

print("Inizializzazione DINOv2 (dinov2_vits14)...")
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(device)
model.eval()

results_dir = os.path.join(base_dir, "results")
os.makedirs(results_dir, exist_ok=True)

# STRUTTURE DATI PER IL PROFILING
layer_times = {f"Block_{i}_Attention": [] for i in range(12)}
layer_times.update({f"Block_{i}_MLP": [] for i in range(12)})
cuda_events = {}

# HOOKS
def get_pre_hook(name):
    def pre_hook(module, input):
        if device.type == "cuda":
            start_evt = torch.cuda.Event(enable_timing=True)
            end_evt = torch.cuda.Event(enable_timing=True)
            start_evt.record()
            cuda_events[name] = (start_evt, end_evt)
        else:
            module.start_time = time.perf_counter()
    return pre_hook

def get_post_hook(name):
    def post_hook(module, input, output):
        if device.type == "cuda":
            if name in cuda_events:
                _, end_evt = cuda_events[name]
                end_evt.record()
        else:
            duration = time.perf_counter() - module.start_time
            layer_times[name].append(duration)
    return post_hook

hook_handles = []
for idx, block in enumerate(model.blocks):
    attn_name = f"Block_{idx}_Attention"
    mlp_name = f"Block_{idx}_MLP"
    hook_handles.append(block.attn.register_forward_pre_hook(get_pre_hook(attn_name)))
    hook_handles.append(block.attn.register_forward_hook(get_post_hook(attn_name)))
    hook_handles.append(block.mlp.register_forward_pre_hook(get_pre_hook(mlp_name)))
    hook_handles.append(block.mlp.register_forward_hook(get_post_hook(mlp_name)))

print(f"Avvio profiling su {len(image_tensors)-10} immagini (+10 warm-up)...")

with torch.no_grad():
    # warm up
    if len(image_tensors) > 0:
      for tensor in image_tensors[:10]:
        _ = model(tensor)
        if device.type == "cuda":
            torch.cuda.synchronize()
        cuda_events.clear()

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    process = psutil.Process(os.getpid())
    start_ram = process.memory_info().rss

    # profiling
    for tensor in image_tensors[10:]:
        _ = model(tensor)
        if device.type == "cuda":
            torch.cuda.synchronize()
            for name, (start_evt, end_evt) in cuda_events.items():
                elapsed_time = start_evt.elapsed_time(end_evt) / 1000.0
                layer_times[name].append(elapsed_time)
            cuda_events.clear()

    end_ram = process.memory_info().rss
    ram_used_mb = (end_ram - start_ram) / (1024 ** 2)

    if device.type == "cuda":
        vram_used_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

for handle in hook_handles: handle.remove()

print("\n--- MEMORY ---")
print(f"RAM using during inference: {ram_used_mb:.2f} MB")
if device.type == "cuda":
    print(f"Picco di VRAM allocata su GPU: {vram_used_mb:.2f} MB")
print("---------------------------\n")

# RISULTATI
final_averages = {name: (sum(times)/len(times) if len(times)>0 else 0.0) for name, times in layer_times.items()}
df_results = pd.DataFrame(list(final_averages.items()), columns=["Layer", f"Media ({device.type}) [s]"])
csv_filename = os.path.join(results_dir, f"results_profiling_{device.type}.csv")
df_results.to_csv(csv_filename, index=False)
print(f"Profiling completato! File CSV salvato in: '{csv_filename}'")
display(df_results)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

results_dir = os.path.join(base_dir, "results")
file_cpu = os.path.join(results_dir, "results_profiling_cpu.csv")
file_gpu = os.path.join(results_dir, "results_profiling_cuda.csv")

if os.path.exists(file_cpu) and os.path.exists(file_gpu):
    df_cpu = pd.read_csv(file_cpu)
    df_gpu = pd.read_csv(file_gpu)

    # Unione dei dati
    df = pd.merge(df_cpu, df_gpu, on="Layer", suffixes=('_CPU', '_GPU'))

    col_cpu = [c for c in df.columns if "cpu" in c.lower()][0]
    col_gpu = [c for c in df.columns if "cuda" in c.lower() or "gpu" in c.lower()][0]

    df["CPU (ms)"] = df[col_cpu] * 1000
    df["GPU (ms)"] = df[col_gpu] * 1000

    # Plotting combinato CPU vs GPU
    x = np.arange(len(df["Layer"]))
    width = 0.35
    fig1, ax1 = plt.subplots(figsize=(14, 6))

    # SX (CPU - Rosso)
    color = 'tab:red'
    ax1.set_xlabel('Layer of model (DINOv2 ViT-Small)', fontweight='bold', labelpad=12)
    ax1.set_ylabel('Time CPU (ms)', color=color, fontweight='bold')
    ax1.bar(x - width/2, df["CPU (ms)"], width, label='CPU', color=color, alpha=0.7)
    ax1.tick_params(axis='y', labelcolor=color)

    # DX (GPU - Blu)
    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Time GPU (ms)', color=color, fontweight='bold')
    ax2.bar(x + width/2, df["GPU (ms)"], width, label='GPU', color=color, alpha=0.7)
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title('Time: CPU vs GPU', fontsize=14, fontweight='bold', pad=15)
    ax1.set_xticks(x)
    ax1.set_xticklabels(df["Layer"], rotation=90)
    fig1.tight_layout()

    # Salvataggio Plot combinato
    plot_path_combined = os.path.join(results_dir, "profiling_combined.png")
    fig1.savefig(plot_path_combined, dpi=300)
    print(f"Grafico combinato salvato con successo in: '{plot_path_combined}'")
    plt.show()

    # Plot singolo per CPU
    fig_cpu, ax_cpu = plt.subplots(figsize=(14, 6))
    ax_cpu.bar(x, df["CPU (ms)"], color='tab:red', alpha=0.7)
    ax_cpu.set_title('Latency per Layer (CPU)', fontsize=14, fontweight='bold', pad=15)
    ax_cpu.set_xlabel('Layer of model (DINOv2 ViT-Small)', fontweight='bold', labelpad=12)
    ax_cpu.set_ylabel('Time CPU (ms)', fontweight='bold')
    ax_cpu.set_xticks(x)
    ax_cpu.set_xticklabels(df["Layer"], rotation=90)
    fig_cpu.tight_layout()
    plot_path_cpu = os.path.join(results_dir, "profiling_cpu.png")
    fig_cpu.savefig(plot_path_cpu, dpi=300)
    print(f"Grafico CPU salvato con successo in: '{plot_path_cpu}'")
    plt.show()

    # Plot singolo per GPU
    fig_gpu, ax_gpu = plt.subplots(figsize=(14, 6))
    ax_gpu.bar(x, df["GPU (ms)"], color='tab:blue', alpha=0.7)
    ax_gpu.set_title('Latency per Layer (GPU)', fontsize=14, fontweight='bold', pad=15)
    ax_gpu.set_xlabel('Layer of model (DINOv2 ViT-Small)', fontweight='bold', labelpad=12)
    ax_gpu.set_ylabel('Time GPU (ms)', fontweight='bold')
    ax_gpu.set_xticks(x)
    ax_gpu.set_xticklabels(df["Layer"], rotation=90)
    fig_gpu.tight_layout()
    plot_path_gpu = os.path.join(results_dir, "profiling_gpu.png")
    fig_gpu.savefig(plot_path_gpu, dpi=300)
    print(f"Grafico GPU salvato con successo in: '{plot_path_gpu}'")
    plt.show()

else:
    print("File CSV non trovati. Assicurati di aver eseguito il profiling sia su CPU che su GPU.")

In [ ]:
total_time_cpu_per_image = df_cpu['Media (cpu) [s]'].sum()
total_time_gpu_per_image = df_gpu['Media (cuda) [s]'].sum()

# Calcolo del tempo medio dei layer per CPU
time_avg_cpu_att = df_cpu[df_cpu['Layer'].str.contains('Attention')]['Media (cpu) [s]'].mean()
time_avg_cpu_mlp = df_cpu[df_cpu['Layer'].str.contains('MLP')]['Media (cpu) [s]'].mean()

# Calcolo del tempo medio dei layer per GPU
time_avg_gpu_att = df_gpu[df_gpu['Layer'].str.contains('Attention')]['Media (cuda) [s]'].mean()
time_avg_gpu_mlp = df_gpu[df_gpu['Layer'].str.contains('MLP')]['Media (cuda) [s]'].mean()

throughput_cpu = 1 / total_time_cpu_per_image
throughput_gpu = 1 / total_time_gpu_per_image

# Calcolo dello Speedup
speedup_total_time = total_time_cpu_per_image / total_time_gpu_per_image
speedup_attention = time_avg_cpu_att / time_avg_gpu_att
speedup_mlp = time_avg_cpu_mlp / time_avg_gpu_mlp

print(f"Latenza media Attention CPU: {time_avg_cpu_att * 1000:.4f} ms")
print(f"Latenza media MLP CPU: {time_avg_cpu_mlp * 1000:.4f} ms")
print(f"Latenza media Attention GPU: {time_avg_gpu_att * 1000:.4f} ms")
print(f"Latenza media MLP GPU: {time_avg_gpu_mlp * 1000:.4f} ms")

print(f"\nTempo medio totale per immagine (CPU): {total_time_cpu_per_image:.4f} secondi")
print(f"Throughput CPU: {throughput_cpu:.2f} immagini/secondo")
print(f"Tempo medio totale per immagine (GPU): {total_time_gpu_per_image:.4f} secondi")
print(f"Throughput GPU: {throughput_gpu:.2f} immagini/secondo")

print(f"\n--- SPEEDUP (GPU vs CPU) ---")
print(f"Speedup Tempo Totale: {speedup_total_time:.2f}x")
print(f"Speedup Layer Attention: {speedup_attention:.2f}x")
print(f"Speedup Layer MLP: {speedup_mlp:.2f}x")